# Acrylic Lichtenberg — From Prescribed Space Charge to Fractal Tree

When a beam-charged acrylic slab is tapped with a grounded nail, the trapped electrons find a path to ground in nanoseconds. What's left behind is a branching 3D tree frozen into the polymer — a permanent record of the discharge dynamics. This notebook walks through the whole event, quasi-statically, in ten stages:

1. **The Material** — pin down the PMMA constants that everything else reads from.
2. **The Slab and the Nail** — define the spatial domain, the boundary conditions, and where the grounded electrode enters.
3. **The Prescribed Charge Cloud** — the one shortcut in this notebook: we write down $\rho(\vec r)$ directly instead of simulating an electron beam.
4. **Calibrating to the Breakdown Threshold** — scale $\rho$ so the slab sits at the fracture-scale regime.
5. **The Background Field** — one FFT gives us $V_\text{free}(\vec r)$.
6. **The Energy Reservoir** — record $U_\text{before}$ so we can measure what the discharge drains.
7. **The Discharge** — grow the dielectric-breakdown tree cell by cell.
8. **Induced Surface Charge** — where does the opposing charge sit?
9. **The Tree in 3D** — look at it.
10. **Fractal Morphology** — does $D_f$ land where real acrylic figures do?

In [ ]:
import io

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from PIL import Image
from scipy.fft import dstn, idstn

## 1. The Material

PMMA (acrylic) is the substrate. The model's entire material footprint lives in two numbers:

- $\epsilon_r$ (relative permittivity) sets how much charge the slab can hold per unit voltage. It enters the Poisson solve and the stored-energy formula.
- $E_b$ (intrinsic dielectric strength) sets the breakdown field the tree must beat to propagate. Anchors the calibration target and the per-step breakdown check.

In [ ]:
# Fundamental constants.
q_e = -1.602176634e-19  # electron charge [C]
eps_0 = 8.8541878128e-12  # vacuum permittivity [F/m]

# PMMA (acrylic) material properties -- only the two that drive the model.
PMMA = {
    "eps_r": 3.4,   # relative permittivity
    "E_b": 2.0e7,   # intrinsic dielectric strength [V/m] (20 kV/mm)
}

eps_r = PMMA["eps_r"]
cm_to_m = 1e-2

## 2. The Slab and the Nail

With the material fixed, we carve out a domain: a 10 × 10 × 4 cm rectangular box discretised on a uniform grid at 1 mm resolution. All six faces are held at $V = 0$ — a grounded Faraday-cage boundary condition.

That BC is a simplification. In a real setup only the back plate would be a true conductor; the other five faces would be dielectric–air interfaces carrying bound polarization charge rather than free induced charge. We use the Faraday-cage BC because:

- It makes the upcoming Poisson solve *exact* with a single FFT (homogeneous Dirichlet on a box is the setup the Discrete Sine Transform diagonalises exactly).
- It captures the correct qualitative physics: induced charge distributed over every surface of the enclosure, summing to $-Q_\text{interior}$ by Gauss's law. We'll extract that induced-charge distribution explicitly later in the notebook as a sanity check.

The nail — our grounded electrode — is driven into the **$-x$ side face**, centred along $y$, at the exact depth where $\rho$ peaks. That's the most aggressive version of the "dielectric barrier reduction" idea from Theory §4.3: the gap between the grounded conductor and the peak-$\rho$ plane collapses to a single grid cell, so the initial breakdown field is maximal. Rather than dropping down into the charge disk from above, the tree will grow *laterally through* it — the classic side-tapped Lichtenberg geometry.

In [ ]:
resolution = 0.1

x_size = 10
y_size = 10
z_size = 4

x = np.arange(0, x_size, resolution)
y = np.arange(0, y_size, resolution)
z = np.arange(0, z_size, resolution)

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
nx_, ny_, nz_ = X.shape

h = resolution * cm_to_m  # grid spacing [m]

## 3. The Prescribed Charge Cloud

Here's the shortcut that names this notebook. The companion implantation notebook samples $\rho$ from beam physics: Katz–Penfold CSDA range picks the mean depth, Landau–Vavilov straggling picks the spread, and the end-state $\rho(\vec r)$ is the stochastic *outcome* of a Monte-Carlo beam simulation.

We skip all of that and just write down the density we want — a Gaussian-depth profile at $z = z_\text{centre}$ (the same depth the CSDA formula would put it) multiplied by a tanh-softened disk in the $xy$-plane (the same lateral footprint the beam would produce). The advantage: we can reshape $\rho$ freely — thicker, thinner, off-centre, annular, multi-layer — and re-run the notebook without touching anything upstream. The disadvantage: the shape is now our *assumption* rather than a derived consequence, and different assumptions give different trees. Treat this cell as the main physical input to the whole downstream story.

In [ ]:
# Shape parameters of the prescribed charge cloud.
z_centre = 2.2    # layer depth [cm]
sigma_z = 0.35    # Gaussian half-thickness in z [cm]
r_disk = 3.5      # lateral disk radius [cm]
edge_soft = 0.6   # softness of the tanh disk edge [cm]

x_c, y_c = 0.5 * x_size, 0.5 * y_size
R_xy = np.sqrt((X - x_c) ** 2 + (Y - y_c) ** 2)

depth_profile = np.exp(-((Z - z_centre) ** 2) / (2.0 * sigma_z ** 2))
disk_window = 0.5 * (1.0 - np.tanh((R_xy - r_disk) / edge_soft))

rho_unnormalised = depth_profile * disk_window  # dimensionless shape

print(f"Cloud centre: ({x_c:.1f}, {y_c:.1f}, {z_centre:.1f}) cm")
print(
    f"Depth sigma = {sigma_z:.2f} cm"
    f"  |  disk radius = {r_disk:.2f} cm"
    f"  (edge {edge_soft:.2f} cm)"
)

### Is this $\rho(\vec r)$ the correct way to prescribe the charge?

The two analytic factors above are not arbitrary shapes — each maps onto a known piece of beam-implantation physics, and together they reproduce the *stationary* charge distribution that a real MeV-electron irradiation leaves behind once transport has thermalised. A quick audit:

**1. Gaussian depth profile.** For MeV electrons fully stopped in PMMA, the depth at which any single electron comes to rest is the sum of a large number of small-angle scattering and energy-loss events. Energy straggling is described exactly by the Landau–Vavilov distribution, which is skewed in the *thin-absorber* limit but collapses to a Gaussian (by the central-limit theorem) once the track length covers many mean free paths — the regime we are in at $z_\text{centre} = 2.2$ cm. The mean depth is set by the CSDA range via the Katz–Penfold empirical fit, and the spread $\sigma_z$ is the range straggling at that depth. So `np.exp(-(Z - z_centre)**2 / (2*sigma_z**2))` is the correct leading-order functional form, not just a convenient bell curve.

**2. Soft-edged disk lateral profile.** A real accelerator beam is scanned (raster) or defocused over a footprint much wider than the individual beam spot. Inside the footprint the implanted areal density is essentially uniform; outside it falls to zero over the divergence scale of the beam. `0.5 * (1 - tanh((R_xy - r_disk)/edge_soft))` is exactly that: a flat interior of radius $r_\text{disk}$ with a smooth transition to zero on the scale `edge_soft`. The softness is not just cosmetic — a hard step would introduce Gibbs-style ringing in the DST-based Poisson solve of §5, so the tanh edge serves numerical hygiene *and* physical fidelity at once.

**3. Separability is justified.** Writing $\rho(x,y,z) = f(z)\,g(r_{xy})$ only works because the depth distribution (set by stopping power, a $z$-only quantity) and the lateral distribution (set by beam optics, an $xy$-only quantity) are physically independent to leading order. They couple weakly through multiple-scattering divergence as the beam slows, but the coupling is a few-percent correction on the field-scale quantities the downstream Poisson solve actually cares about.

**4. What we still approximate — and why it's tolerable here.**

- *Symmetric Gaussian in depth.* Real range distributions carry a mild negative skew (longer shallow tail, sharper deep cutoff). The companion Monte-Carlo notebook captures that skew explicitly; here we take the symmetric Gaussian that straggling theory gives in the asymptotic limit, accurate to within a few percent of the profile integral.
- *No shot noise.* $\rho$ is smooth, not a sum of $N$ individual electron contributions. Since we coarse-grain to field quantities on a cm-scale grid where $N \sim 10^{13}$ electrons per voxel, Poisson fluctuations are negligible (relative amplitude $\sim N^{-1/2} \sim 10^{-6}$).
- *Homogeneous PMMA.* No dielectric heterogeneity is baked into $\rho$. The sister Wood-Lichtenberg notebook *does* fold in a log-normal heterogeneity field because wood's variability is the dominant branching seed; in bulk charge-implanted acrylic, nail geometry and field-dependent breakdown strength dominate instead, so a clean $\rho$ is the appropriate starting point.

**Verdict.** The prescribed $\rho$ is the correct quasi-static charge distribution for this problem: it is the analytic limit of the physical implantation process, regularised enough for a spectral Poisson solve, and parameterised so that any reshape — tilt, offset, annulus, multi-layer — reduces to editing the four numbers in the cell above.

## 4. Calibrating to the Breakdown Threshold

A shape without a magnitude doesn't discharge. We scale $\rho$ so that its equivalent surface charge density

$$\sigma \;=\; \frac{1}{A_\text{surf}}\int \rho(\vec r)\,dV$$

lands at a chosen fraction of the Gauss-law breakdown density $\sigma_b = \varepsilon_0\,\varepsilon_r\,E_b$ (Theory §4.2). This fraction is the central dial of the whole notebook:

- `sigma_fraction` $\ll 1$ — **sub-critical.** The bulk field is comfortably below $E_b$ and the slab sits quiescently, holding its charge for hours. The nail-tip enhancement can still nucleate a tree, but the discharge is small.
- `sigma_fraction` $= 1$ — **fracture-scale.** The bulk field inside the charge layer reaches $E_b$ on its own; the slab is one nail-tap away from self-discharge. This is the regime that produces the big, dramatic trees seen in real charge-implanted acrylic.
- `sigma_fraction` $> 1$ — **super-critical.** The slab cracks without provocation; the simulation is no longer really valid (the charge distribution wouldn't be stable in the first place).

We run at `sigma_fraction = 1.0`, the fracture-scale regime. Everything downstream scales with this knob: the stored energy scales roughly as $\sigma^2$ (quadratic in field), and the total charge as $\sigma$, so dialling it down by half roughly quarters the energy reservoir and shrinks the resulting tree visibly.

In [ ]:
sigma_fraction = 1.0  # 1.0 = at breakdown; 0.5 = sub-critical

A_surf_m2 = (x_size * cm_to_m) * (y_size * cm_to_m)
dV_cell_m3 = h ** 3
sigma_b = eps_0 * PMMA["eps_r"] * PMMA["E_b"]
sigma_target = sigma_fraction * sigma_b
Q_total = -sigma_target * A_surf_m2  # electrons -> negative

# Scale the unit-less shape so the integrated charge matches Q_total.
shape_integral = rho_unnormalised.sum() * dV_cell_m3  # [m^3]
rho_peak = Q_total / shape_integral  # [C/m^3] where shape == 1
rho = rho_peak * rho_unnormalised

print(f"sigma_b (breakdown)        = {sigma_b:.3e} C/m^2")
print(
    f"sigma_target ({sigma_fraction:.0%} of sigma_b)"
    f" = {sigma_target:.3e} C/m^2"
)
print(f"Q_total                    = {Q_total:.3e} C ({Q_total * 1e6:.2f} uC)")
print(f"Peak |rho|                 = {abs(rho).max():.3e} C/m^3")
print(
    f"Equivalent packed electrons / cm^3 (peak):"
    f" {abs(rho).max() * 1e-6 / abs(q_e):.2e}"
)

## 5. The Background Field

With $\rho$ now fully specified, the potential it creates satisfies Poisson's equation (Theory §2.1 sets up this equation and its boundary-value formulation):

$$-\nabla^2 V_\text{free} \;=\; \frac{\rho}{\varepsilon_0\,\varepsilon_r},\qquad V_\text{free}=0\ \text{on every wall of the box.}$$

The medium is uniform ($\varepsilon_r$ is the same everywhere inside the slab) and the boundary condition is homogeneous Dirichlet on a rectangular box — exactly the setup that the Type-I Discrete Sine Transform diagonalises. The solve collapses into three elementwise operations:

1. Forward DST of the source $\rho / (\varepsilon_0\,\varepsilon_r)$.
2. Divide by the eigenvalues of the negative discrete Laplacian,

$$\lambda_{ijk}\;=\;\frac{2}{h^2}\left(3 - \cos\frac{i\pi}{n_x+1} - \cos\frac{j\pi}{n_y+1} - \cos\frac{k\pi}{n_z+1}\right).$$

3. Inverse DST.

Exact at the grid level, no iteration, a fraction of a second at the default resolution.

What we get out is the *background* field — the potential due to $\rho$ alone with the box walls grounded. It does not yet satisfy the constraint that will come with the tree ($V = 0$ on tree cells too), because the tree doesn't exist yet. Once the discharge begins, we'll add a harmonic correction $V_\text{induced}$ so that $V_\text{total} = V_\text{free} + V_\text{induced}$ vanishes on both the box walls *and* on every tree cell.

Splitting the problem this way — the source term handled once by FFT, the moving tree boundary handled per-step by Jacobi relaxation — is what makes the main discharge loop fast.

In [ ]:
def poisson_dirichlet(source, h):
    """Solve -div grad V = source on a box with V = 0 on all six faces.

    Grid nodes are treated as interior points with virtual zero-boundary
    nodes one spacing outside the array. The Type-I Discrete Sine
    Transform diagonalises the discrete Laplacian under this BC, so the
    solve is just a forward DST, an elementwise divide, and an inverse
    DST.
    """
    nx, ny, nz = source.shape
    f_hat = dstn(source, type=1, norm="ortho")

    kx = np.arange(1, nx + 1)
    ky = np.arange(1, ny + 1)
    kz = np.arange(1, nz + 1)
    lam_x = (2.0 / h ** 2) * (1.0 - np.cos(np.pi * kx / (nx + 1)))
    lam_y = (2.0 / h ** 2) * (1.0 - np.cos(np.pi * ky / (ny + 1)))
    lam_z = (2.0 / h ** 2) * (1.0 - np.cos(np.pi * kz / (nz + 1)))

    Lx = lam_x[:, None, None]
    Ly = lam_y[None, :, None]
    Lz = lam_z[None, None, :]
    V_hat = f_hat / (Lx + Ly + Lz)

    return idstn(V_hat, type=1, norm="ortho")


source_free = rho / (eps_0 * eps_r)  # -div grad V = rho/eps
V_free = poisson_dirichlet(source_free, h)

V_peak = float(np.abs(V_free).max())
grad_z = np.gradient(V_free, h, axis=2)
E_peak_implied = float(np.abs(grad_z).max())

print(f"V_free range: [{V_free.min():.3e}, {V_free.max():.3e}] V")
print(f"Peak |V_free|: {V_peak:.3e} V")
print(
    f"Peak bulk |E_z| = {E_peak_implied:.3e} V/m"
    f"  ({E_peak_implied / PMMA['E_b'] * 100:.1f}% of E_b)"
)

## 6. The Energy Reservoir

The slab is now a capacitor charged to the brink. Its stored electrostatic energy (Theory §4.2),

$$U \;=\; \tfrac{1}{2}\,\varepsilon_0\,\varepsilon_r\,\int |\vec E|^2 \, dV \;=\; \tfrac{1}{2}\,\varepsilon_0\,\varepsilon_r\,\int |\nabla V|^2 \, dV,$$

is the reservoir the discharge will drain. We record `U_before` here from $V_\text{free}$ alone; once the tree has finished growing we will recompute it from $V_\text{total}$ and report the difference as `U_released`.

One caveat that matters when reading the numbers at the end: "released" here means only that the *field-energy state* of the system has dropped. Our quasi-static model does not track where the energy actually went. In reality the difference is dissipated as plasma Joule heating, optical flash, acoustic shock, and mechanical fracture along the channel walls. The electrostatic bookkeeping captures the reservoir that sourced those physical outflows without modelling them directly.

In [ ]:
def electrostatic_energy(V_field, h, eps_r):
    """Return U = 0.5 * eps_0 * eps_r * integral |grad V|^2 dV.

    Central differences for each component of grad V, then a box
    integration over the whole grid.
    """
    Ex = -(np.roll(V_field, -1, 0) - np.roll(V_field, 1, 0)) / (2.0 * h)
    Ey = -(np.roll(V_field, -1, 1) - np.roll(V_field, 1, 1)) / (2.0 * h)
    Ez = -(np.roll(V_field, -1, 2) - np.roll(V_field, 1, 2)) / (2.0 * h)
    grad_sq_sum = (Ex * Ex + Ey * Ey + Ez * Ez).sum()
    return 0.5 * eps_0 * eps_r * grad_sq_sum * h ** 3


U_before = electrostatic_energy(V_free, h, PMMA["eps_r"])
print(f"Stored electrostatic energy (pre-discharge):  U = {U_before:.3f} J")

## 7. The Discharge — Dielectric-Breakdown-Model Tree Growth

Now the event itself. Starting from a single grounded cell at the nail, the tree grows one cell per iteration until it runs out of stored charge in reach. The mechanism is Niemeyer–Pietronero–Wiesmann dielectric-breakdown modelling — a stochastic rule that generalises the diffusion-limited-aggregation construction of Theory §1.2 by weighting each growth step with the local electric field instead of uniform walker arrival, and that is physically motivated by the tip-splitting instability of Theory §3.1. We add two upgrades beyond the textbook version:

- **26-connectivity** instead of 6. Face-only neighbours force every branch onto one of three axes, producing a staircase that doesn't look like a real streamer. With face + edge + corner neighbours we get 13 distinct line directions and much more natural-looking branches.
- **Field-dependent charge absorption.** When the tree advances into a cell, it drags along a small halo of surrounding charge — the plasma channel's physical width plus the mechanical-fracture zone around it. Both grow with the local field, so we scale the absorption radius by the edge field that drove the step.

The rest of this section walks through each piece in order:

1. Set the parameters — the physics dials ($\eta$, $\beta_\text{tip}$) and the numerical budget (max steps, relaxation sweeps).
2. Build the connectivity table and the field-dependent absorption rule.
3. Place the nail and initialise the simulation state.
4. Define the three helper functions the inner loop uses.
5. Run the growth loop and do a final Poisson re-solve once it stops.

In [ ]:
# Step 7.1 — DBM parameters.

# Breakdown kinetics.
eta = 3.0                                 # dendritic regime
beta_tip = 50.0                           # sub-grid nail-tip enhancement
E_threshold = PMMA["E_b"] / beta_tip      # edge-field breakdown threshold

# Numerical budget.
max_steps = 3000                          # cap on tree-growth iterations
relax_iters = 8                           # Jacobi sweeps per growth step
refresh_every = 100                       # Poisson re-solve on depleted rho

print(f"eta = {eta}  (>1 tip-seeking, 1 neutral, <1 blob-filling)")
print(f"E_b = {PMMA['E_b']:.2e} V/m,  beta_tip = {beta_tip}")
print(f"Edge-field threshold = E_b / beta_tip = {E_threshold:.2e} V/m")
print(f"Budget: up to {max_steps} growth steps,"
      f" {relax_iters} Jacobi sweeps each,"
      f" Poisson refresh every {refresh_every} steps")

### What each DBM constant means, and why the value

The cell above packs six numbers into three lines. They divide cleanly into two groups: **physics dials** ($\eta$, $\beta_\text{tip}$, and the derived $E_\text{threshold}$), and **numerical budget** (`max_steps`, `relax_iters`, `refresh_every`). Changing a physics dial reshapes the tree; changing a numerical knob only changes how fast or how accurately the loop runs.

**`eta = 3.0` — the Niemeyer–Pietronero–Wiesmann branching exponent (Theory §1.2, §3.1).** The probability a candidate cell breaks down next scales as $|E_\text{edge}|^\eta$. $\eta > 1$ rewards the cell with the strongest field disproportionately, so the tree races to high-field tips (dendritic regime); $\eta = 1$ is neutral DLA-like; $\eta < 1$ is space-filling. We pick $\eta = 3$ because this is the value that reproduces the fractal dimension $D_f \approx 1.7$ reported for real charge-implanted PMMA figures (Theory §4.5).

**`beta_tip = 50.0` — yes, this is a constant, and it represents sub-grid geometry (Theory §2.3).** The grid spacing is $h = 1$ mm, but a real nail has a tip radius of order 10–100 μm. The actual field at the physical tip is therefore much larger than anything a 1-mm grid can resolve. Rather than re-mesh the whole domain around the nail, we bake in a lumped multiplier. For a hemispherical tip of radius $r_\text{tip}$ against a plane at distance $d$, classical electrostatics gives an enhancement $\approx d/r_\text{tip}$; with $d \sim 1$ cm and $r_\text{tip} \sim 0.02$ cm this lands at $\beta \sim 50$. Holding it constant is defensible because it only matters at the **first few growth steps**: once the tree has advanced a handful of cells, its own leading tip is resolved on the grid at its natural radius of curvature, and no sub-grid correction is needed. In effect, `beta_tip` is a one-shot nucleation aid for the first cell leaving the nail.

**`E_threshold = PMMA["E_b"] / beta_tip` — the coarse-grid trigger for breakdown.** Not an independent parameter: it's fully determined by the two above it. The derivation is one line: physically breakdown happens when $E_\text{real} \geq E_b$. Near the nail, $E_\text{real} \approx \beta_\text{tip} \cdot E_\text{grid}$. Solving for the grid-scale field gives $E_\text{grid} \geq E_b / \beta_\text{tip} \equiv E_\text{threshold}$. Any candidate cell whose edge field falls below this value is physically not yet capable of sustaining a streamer, and the growth loop exits when *every* remaining candidate fails this test — the physically meaningful stopping condition ("the discharge has run out of chargeable paths"). It's also the floor used in the absorption-radius formula in Step 7.2: `ratio = max(E_edge, E_threshold) / E_threshold` so that $r_\text{abs}$ never collapses below its minimum.

**`max_steps = 3000` — numerical safety cap, not physics.** The $E_\text{threshold}$ check is the real stopping condition; `max_steps` is a seatbelt in case the run somehow never satisfies it (pathologically large `sigma_fraction`, a bug, or too coarse a grid). At $h = 1$ mm, a 3000-cell tree is already a substantial structure — more than what a real discharge produces at our parameters — so bailing out here indicates something is wrong upstream.

**`relax_iters = 8` — Jacobi sweeps on $V_\text{induced}$ per growth step.** Each step, only *one* cell is added to the tree, so $V_\text{induced}$ only has to adjust around that one new constraint. Warm-starting from the previous step's solution, eight sweeps is enough to re-converge the Laplace correction to within the noise of the candidate-field comparison. Fewer and the stale $V_\text{induced}$ biases the edge-field rankings; more and the loop slows with no accuracy gain. The final 30 sweeps after the loop (Step 7.5) bring it to energy-accurate convergence for the post-discharge bookkeeping.

**`refresh_every = 100` — how often to re-solve the full FFT Poisson problem on the depleted $\rho$.** Between refreshes, $V_\text{free}$ is frozen while the loop chips away at `rho_remaining` via `absorb_sphere`; strictly, $V_\text{free}$ should track those changes continuously. Re-solving every step would make the loop ~50× slower because the FFT dominates per-step cost. Re-solving every 100 steps keeps the FFT amortised to a small fraction of total runtime while the drift stays within a few percent of $V_\text{free}$, which is well below the noise from the stochastic DBM pick itself. Lower values cost more for no visible change; higher values start to visibly under-grow the tree because depleted regions keep pulling charge they no longer contain.

**Summary of the dependency graph.** Of these six numbers, only $\eta$, $\beta_\text{tip}$, and $\sigma_\text{fraction}$ (set back in §4) are truly independent physics choices. $E_\text{threshold}$ is derived from $\beta_\text{tip}$ and $E_b$. The three numerical knobs (`max_steps`, `relax_iters`, `refresh_every`) would ideally be infinite / 1 / 1 in a perfect world; their actual values are chosen so the simulation runs in minutes instead of hours without the output changing measurably.

### Step 7.2 — Connectivity and Absorption Radius

Two geometric rules that every iteration of the main loop relies on.

**26-connectivity.** In a 3D grid, every interior cell has 26 neighbours: 6 face-adjacent (distance $h$), 12 edge-adjacent (distance $\sqrt{2}\,h$), and 8 corner-adjacent (distance $\sqrt{3}\,h$). Using all 26 instead of only the 6 face-neighbours gives the tree 13 distinct line directions to branch along, so it no longer looks like it was drawn on graph paper. We pre-compute the offsets and their physical distances once.

**Field-dependent absorption radius.** Each time the tree claims a new cell, we zero the charge in a small cube around it — physically, the charge in that halo has found its path to ground. How big the cube should be depends on how hot the channel is at that point. Plasma + fracture physics gives two limits:

- Drift radius (electrons drifting into the channel under the attractive field): $r_\text{drift} \propto |\vec{E}|$.
- Mechanical-fracture width (cracking driven by energy density): $r_\text{frac} \propto |\vec{E}|^2$.

We interpolate the two with a single power-law exponent $\alpha = 0.5$ and clamp the result into $[r_\text{min},\ r_\text{max}] = [1,\ 3]$ cells:

$$r_\text{abs}(|\vec{E}|)\;=\;\text{clip}\!\left(r_\text{min}\left(\frac{|\vec{E}|}{E_\text{threshold}}\right)^{\alpha},\ r_\text{min},\ r_\text{max}\right).$$

Hot-field segments near the nail therefore sweep a wider halo; marginal-threshold tips at the fringe absorb only the one cell they occupy.

In [ ]:
# Step 7.2 — Connectivity table and absorption-radius function.

offsets_26 = [
    (di, dj, dk)
    for di in (-1, 0, 1)
    for dj in (-1, 0, 1)
    for dk in (-1, 0, 1)
    if not (di == 0 and dj == 0 and dk == 0)
]

dist_26_m = {
    o: float(np.sqrt(o[0] ** 2 + o[1] ** 2 + o[2] ** 2)) * h
    for o in offsets_26
}

r_abs_min = 1       # plasma-channel scale (sub-grid at h = 1 mm)
r_abs_max = 3       # 7x7x7 cube; wider is meaningless at this resolution
r_abs_alpha = 0.5   # between drift (alpha=1) and fracture (alpha=2)


def r_absorb_from_edge(e_edge):
    """Return the absorption half-width (in cells) for a given edge field."""
    ratio = max(float(e_edge), E_threshold) / E_threshold
    return int(np.clip(
        round(r_abs_min * ratio ** r_abs_alpha),
        r_abs_min,
        r_abs_max,
    ))


print(f"26-connectivity: {len(offsets_26)} neighbours per cell")
print(f"  6 face (d=h), 12 edge (d=sqrt(2)*h), 8 corner (d=sqrt(3)*h)")
print(
    f"r_abs = clip({r_abs_min}*(|E|/E_th)^{r_abs_alpha},"
    f" {r_abs_min}, {r_abs_max}) cells"
)

### Step 7.3 — Nail Placement and Initial State

The simulation starts with exactly one cell in the tree: the nail itself. Everything else is bookkeeping.

**Nail position.** The grounded electrode goes into the $-x$ face, centred in $y$, at the depth where $\rho$ peaks — grid index `(0, ny // 2, k_centre)`. Because the nail sits on a box wall (already pinned to $V = 0$ by the Faraday-cage boundary) *and* is a grounded tree cell (also $V = 0$), the initial condition is trivially consistent — no transient needs to die off.

**Persistent state the loop will mutate.**

- `tree` — boolean mask of which cells are in the discharge tree.
- `V_induced` — harmonic correction field; keeps $V_\text{total} = V_\text{free} + V_\text{induced}$ pinned to 0 on all tree cells while satisfying $\nabla^2 V_\text{induced} = 0$ elsewhere. Pre-set at the nail so the sum cancels.
- `tree_step` — integer label recording when each cell was added (used for colour in the 3D viz).
- `parent_map` — dict `{child_cell: parent_cell}`, recorded at growth time so the rendered branches follow the actual 26-connectivity edges taken.
- `rho_remaining` — mutable copy of the charge density that shrinks each time the tree absorbs a halo.
- `candidates` — dict `{cell: (min_edge_distance, best_parent)}` of every non-tree cell currently touching the tree. Built once with the nail's neighbours; incrementally updated each step.
- `Q_absorbed_running`, `r_abs_history` — running diagnostics.

In [ ]:
# Step 7.3 — Nail placement and initial state.

k_centre = int(round(z_centre / resolution))
nail_idx = (0, ny_ // 2, k_centre)

print(
    f"Nail at grid {nail_idx}"
    f" = ({nail_idx[0] * resolution:.2f},"
    f" {nail_idx[1] * resolution:.2f},"
    f" {nail_idx[2] * resolution:.2f}) cm"
)

tree = np.zeros_like(V_free, dtype=bool)
tree[nail_idx] = True

V_induced = np.zeros_like(V_free)
V_induced[nail_idx] = -V_free[nail_idx]

tree_step = np.full(V_free.shape, -1, dtype=np.int32)
tree_step[nail_idx] = 0

parent_map = {}
r_abs_history = []

rho_remaining = rho.copy()
Q_absorbed_running = 0.0

candidates = {}

print(f"Initial tree: 1 cell  |  rho_remaining: {rho_remaining.shape} array")

### Step 7.4 — Helper Functions

Three small functions that the growth loop will call on every iteration. Defining them separately keeps the main loop readable.

- **`register_candidates_from(p)`** — given a newly-added tree cell $p$, look at its 26 neighbours. For each neighbour that isn't already in the tree, record it as a candidate along with the distance to $p$ and the parent $p$ itself. If that neighbour is already a candidate of some *earlier* tree cell, keep whichever parent gives the shorter distance — the main loop uses that shortest distance when computing the edge field. Called once to seed from the nail, and again on every growth step.

- **`laplace_sweep(V_ind, tree_mask, V_f)`** — one Jacobi iteration for the harmonic correction $V_\text{induced}$. Interior cells get replaced by the mean of their six face-neighbours, boundary cells stay pinned to 0 (box Dirichlet), and tree cells are forced to $-V_\text{free}$ so that $V_\text{total} = 0$ on the tree. Running this `relax_iters` times per growth step lets $V_\text{induced}$ catch up with the tree's new shape.

- **`absorb_sphere(i, j, k, r_cells)`** — zero `rho_remaining` in a cube of half-width `r_cells` around cell $(i, j, k)$. Returns the charge that was inside that cube (summed before zeroing), so the loop can tally total charge flushed to ground.

After defining the helpers, we seed `candidates` from the nail and we're ready to iterate.

In [ ]:
# Step 7.4 — Helper functions used inside the main loop.


def register_candidates_from(p):
    """Add p's 26 neighbours to `candidates` (skipping tree cells).

    For each neighbour, remember the shortest edge length and the parent
    that produces it.
    """
    i, j, k = p
    for off in offsets_26:
        ni, nj, nk = i + off[0], j + off[1], k + off[2]
        if not (0 <= ni < nx_ and 0 <= nj < ny_ and 0 <= nk < nz_):
            continue
        n = (ni, nj, nk)
        if tree[n]:
            continue
        d = dist_26_m[off]
        cur = candidates.get(n)
        if cur is None or d < cur[0]:
            candidates[n] = (d, p)


def laplace_sweep(V_ind, tree_mask, V_f):
    """One Jacobi sweep on V_ind (6-point stencil).

    Interior cells relax toward the mean of their face-neighbours; box
    walls stay at 0; tree cells are pinned to -V_f so that V_total = 0.
    """
    V_new = np.zeros_like(V_ind)
    V_new[1:-1, 1:-1, 1:-1] = (
        V_ind[:-2, 1:-1, 1:-1] + V_ind[2:, 1:-1, 1:-1]
        + V_ind[1:-1, :-2, 1:-1] + V_ind[1:-1, 2:, 1:-1]
        + V_ind[1:-1, 1:-1, :-2] + V_ind[1:-1, 1:-1, 2:]
    ) / 6.0
    V_new[tree_mask] = -V_f[tree_mask]
    V_ind[:] = V_new


def absorb_sphere(i, j, k, r_cells):
    """Zero rho_remaining in a cube of half-width r_cells and return dQ."""
    i0, i1 = max(0, i - r_cells), min(nx_, i + r_cells + 1)
    j0, j1 = max(0, j - r_cells), min(ny_, j + r_cells + 1)
    k0, k1 = max(0, k - r_cells), min(nz_, k + r_cells + 1)
    sub = rho_remaining[i0:i1, j0:j1, k0:k1]
    dq = sub.sum() * dV_cell_m3
    sub[:] = 0.0
    return dq


# Seed candidates with the nail's 26 neighbours.
register_candidates_from(nail_idx)

print(f"Seeded {len(candidates)} initial candidates from the nail.")

### Step 7.5 — The Growth Loop

Each iteration does exactly the same six things:

1. **Relax $V_\text{induced}$** by `relax_iters` Jacobi sweeps. Warm-starting from the previous solution means only a handful of sweeps are needed to stay converged.

2. **Compute the edge field** for every candidate: $|\vec{E}_\text{edge}| = |V_\text{total}[c]| / d(p \to c)$, where $p$ is the best (shortest-edge) parent recorded in `candidates`. Candidates whose edge field falls below $E_b / \beta_\text{tip}$ are discarded — no local breakdown possible there.

3. **Sample the next cell** with probability proportional to $|\vec{E}_\text{edge}|^{\eta}$. At $\eta = 3$, a candidate at twice the threshold gets $2^3 = 8\times$ the weight of one at threshold, so the tree strongly prefers tip cells where the field concentrates.

4. **Commit the chosen cell**: mark it as tree, snap $V_\text{induced}$ there to $-V_\text{free}$ so the total stays 0, record the parent, and re-seed candidates from the new cell's neighbours.

5. **Absorb charge** in a cube of half-width $r_\text{abs}$ around the new cell, sized by the edge field that drove the step.

6. **Refresh $V_\text{free}$** every `refresh_every` steps by re-solving the FFT Poisson problem on the depleted $\rho$. Between refreshes $V_\text{free}$ drifts slightly out of sync, but the expensive solve is amortised.

The loop exits early when (i) there are no candidates left (the tree has hit every cell adjacent to it — pathological, shouldn't happen on a big enough grid), or (ii) every remaining candidate's edge field is below threshold (the discharge has run out of chargeable paths). The second is the physically meaningful stopping condition.

After the loop we re-solve Poisson one last time on the final $\rho$ and run 30 extra Jacobi sweeps, so the energy comparison that follows uses the fully-converged state rather than the mid-step snapshot.

In [ ]:
# Step 7.5 — The main growth loop.

rng = np.random.default_rng(0)

for step in range(1, max_steps + 1):

    # 1. Relax V_induced on the current tree shape.
    for _ in range(relax_iters):
        laplace_sweep(V_induced, tree, V_free)

    if not candidates:
        print(f"step {step}: no candidates remain, stopping.")
        break

    # 2. Edge field on every candidate.
    cells = list(candidates.keys())
    cell_ij = np.array(cells, dtype=np.int64)
    d_arr = np.array([candidates[c][0] for c in cells])
    V_tot = (
        V_free[cell_ij[:, 0], cell_ij[:, 1], cell_ij[:, 2]]
        + V_induced[cell_ij[:, 0], cell_ij[:, 1], cell_ij[:, 2]]
    )
    e_edge = np.abs(V_tot) / d_arr

    valid = e_edge > E_threshold
    if not valid.any():
        print(
            f"step {step}: all candidate edge-fields below"
            f" {E_threshold:.2e} V/m -- discharge stalls."
        )
        break

    # 3. Sample the next cell with weight |E_edge|^eta.
    e_valid = e_edge[valid]
    w = e_valid ** eta
    pick = rng.choice(np.where(valid)[0], p=w / w.sum())
    new_pt = tuple(int(v) for v in cell_ij[pick])
    parent = candidates[new_pt][1]
    e_here = float(e_edge[pick])

    # 4. Commit the new cell and re-seed candidates.
    tree[new_pt] = True
    V_induced[new_pt] = -V_free[new_pt]
    tree_step[new_pt] = step
    parent_map[new_pt] = parent
    del candidates[new_pt]
    register_candidates_from(new_pt)

    # 5. Absorb charge with a field-dependent radius.
    r_cells = r_absorb_from_edge(e_here)
    r_abs_history.append(r_cells)
    Q_absorbed_running += absorb_sphere(*new_pt, r_cells)

    # 6. Periodic FFT Poisson refresh on the depleted rho.
    if step % refresh_every == 0:
        V_free[:] = poisson_dirichlet(
            rho_remaining / (eps_0 * eps_r), h
        )
        V_induced[tree] = -V_free[tree]

    if step % 200 == 0:
        frac = Q_absorbed_running / Q_total if Q_total != 0 else 0.0
        r_avg = float(np.mean(r_abs_history[-200:]))
        print(
            f"step {step:4d}"
            f"  |  tree {int(tree.sum()):4d}"
            f"  |  cands {len(candidates):4d}"
            f"  |  Q_abs {Q_absorbed_running * 1e6:+.3f} uC"
            f" ({frac * 100:5.1f}%)"
            f"  |  <r_abs>_200 {r_avg:.2f}"
        )

# Final sync: Poisson on the final rho + extra relaxation for energy accuracy.
V_free[:] = poisson_dirichlet(rho_remaining / (eps_0 * eps_r), h)
V_induced[tree] = -V_free[tree]
for _ in range(30):
    laplace_sweep(V_induced, tree, V_free)

r_abs_arr = np.array(r_abs_history) if r_abs_history else np.array([0])
print(f"\nDone. Tree: {int(tree.sum())} cells.")
print(
    f"Charge flushed: {Q_absorbed_running * 1e6:+.3f} uC"
    f" ({Q_absorbed_running / Q_total * 100:.1f}% of initial)."
)
print(
    f"Absorption radius across {len(r_abs_arr)} steps:"
    f"  min={r_abs_arr.min()},"
    f"  mean={r_abs_arr.mean():.2f},"
    f"  max={r_abs_arr.max()}"
)

In [ ]:
V_total_final = V_free + V_induced
U_after = electrostatic_energy(V_total_final, h, PMMA["eps_r"])
U_released = U_before - U_after

print(f"U_before = {U_before:.3f} J")
print(f"U_after  = {U_after:.3f} J")
print(
    f"Released = {U_released:.3f} J"
    f"  ({U_released / U_before * 100:.1f}% of stored)"
)
print(
    f"Charge flushed to ground: {Q_absorbed_running * 1e6:+.3f} uC"
    f"  ({Q_absorbed_running / Q_total * 100:.1f}% of initial charge)"
)

## 8. Aftermath (a) — Induced Surface Charge on Every Wall

The discharge is over. The tree and every box wall are at $V = 0$; the bulk $\rho$ has been partially depleted by absorption. What remains on the box walls is an induced-charge layer — the counter-charge that Gauss's law (Theory §2.1) demands when you enclose a free-charge distribution in grounded conductors.

That induced charge is *not* concentrated on a back plate. It distributes over all six walls in proportion to the local normal field:

$$\sigma_\text{ind}(x, y) \;=\; -\,\varepsilon_0\,\varepsilon_r\,\partial_n V\big|_\text{wall},$$

with the total pinned by Gauss to

$$\oint \sigma_\text{ind}\, dA \;=\; -\,Q_\text{interior}.$$

A centred cloud gives roughly balanced walls; an off-centre cloud tilts the distribution toward the nearest wall. Either way, every number in this section is proportional to the bulk charge and spread across the whole enclosure.

**Physical caveat.** On a real open-bench Lichtenberg setup only the metal plate under the slab is a true conductor; the top and side faces of the acrylic are dielectric–air interfaces and carry *bound polarization charge* $\sigma_b = \varepsilon_0\,(\varepsilon_r - 1)\,\vec E\cdot\hat n$ instead of free charge on a conductor. Bound charge is smaller by a factor $(\varepsilon_r - 1)/\varepsilon_r \approx 0.71$ for PMMA but still distributed over every surface and still proportional to $\rho$. Modelling the bound-charge version would require breaking the uniform-medium FFT solver and switching to an iterative mixed-BC Poisson solve. The Faraday-cage BC keeps the solve exact and captures the correct qualitative picture: induced charge *everywhere*, summing to $-Q_\text{interior}$.

**The three steps below:**

1. Compute $\sigma_\text{ind}$ on each of the six walls from a one-sided derivative of $V_\text{total}$.
2. Integrate $\sigma_\text{ind}$ over each face to get per-face charges, then verify Gauss's law on the total.
3. Render all six faces on a shared colour scale so the distribution is directly comparable.

### Why the Dirichlet boundary condition already contained this physics

A natural question at this point: if the induced surface charge exists and produces a real field, why wasn't it an input to the tree-growth simulation in §7? Wouldn't the discharge dynamics come out different?

The answer is that the induced charge *was* active the whole time — it just lived implicitly inside the boundary condition. Electrostatics with grounded conductors admits two fully equivalent problem statements:

1. **Poisson + Dirichlet.** Solve $-\nabla^2 V = \rho/(\varepsilon_0\varepsilon_r)$ with $V = 0$ on the conductor surface. This is what `poisson_dirichlet` and `laplace_sweep` do.
2. **Coulomb with explicit $\sigma$.** Place the interior $\rho$ *and* an unknown surface-charge density $\sigma$ on the conductor, and solve for the potential in free space by direct integration. Then require the total $V$ on the conductor surface to equal zero. That condition determines $\sigma$.

Both formulations produce **identical interior fields**. Physically this is because a grounded conductor responds to any applied field by pulling charge out of ground until its own surface potential lands at zero. "$V = 0$ on the conductor" is *the definition* of "there is enough induced charge on the conductor to cancel the applied potential there." You don't need to know $\sigma$ in advance to enforce $V = 0$; conversely, once $V = 0$ is enforced, you have implicitly specified $\sigma$.

**Why the two are equivalent — one line of Gauss's law.** Take a pillbox straddling the conductor surface with area $dA$ and vanishing thickness. Just *outside* the conductor the field is $\vec E$; just *inside*, the field of a conductor in electrostatic equilibrium is zero. The pillbox encloses surface charge $\sigma\,dA$, so Gauss's law gives

$$\vec E \cdot \hat n_\text{outward} = \frac{\sigma}{\varepsilon_0\varepsilon_r} \quad \Longrightarrow \quad \sigma = -\varepsilon_0\varepsilon_r\,\frac{\partial V}{\partial n}$$

(the minus sign comes from $\vec E = -\nabla V$). This is literally the formula Step 8.1 uses.

**So what was §7 actually doing?** Every field value the DBM loop looked at — the edge field $|E|$ on every candidate cell, the $V_\text{induced}$ sweeps enforcing $V = 0$ on the tree, the stored-energy integrand in §6 — was computed from a potential whose boundary values were pinned to zero. That pinning *is* the induced charge, re-expressed as a constraint instead of a source term. The two descriptions agree to machine precision because they are mathematically identical, not approximations of each other.

§8 is not fixing an omission. It's taking the converged $V_\text{total}$ the solver has already produced and reading off the induced $\sigma$ that the solver implicitly placed on each wall, then verifying via Gauss's law on the closed box that the bookkeeping is self-consistent. The same logic applies to the tree surface during growth: Jacobi relaxation was implicitly maintaining induced charge on the tree channel, which is why candidate cells near the tree see their edge fields drop once the tree arrives (screening) — the physical mechanism that eventually stalls the discharge.

### Step 8.1 — The Formula at Each Wall

The free surface charge density on a grounded conductor is

$$\sigma_\text{ind} \;=\; -\,\varepsilon_0\,\varepsilon_r\,\frac{\partial V}{\partial n}\bigg|_\text{wall},$$

where $\partial_n$ is the derivative along the *outward* normal of that wall. In our discretisation every box wall sits one virtual grid spacing outside the array (the DST solver's Dirichlet BC), with $V_\text{wall} = 0$. A one-sided finite difference into the interior therefore reduces to $V_\text{interior} / h$ (or $-V_\text{interior}/h$, depending on face orientation).

We compute all six face arrays in one block below. Signs are arranged so that a negative value of $\sigma_\text{ind}$ means electrons are piled on that wall — the expected result, since the bulk $\rho$ is itself negative.

In [ ]:
# Step 8.1 — Compute sigma_ind on all six walls.

V_total_final = V_free + V_induced

# V = 0 on each wall, so the one-sided difference reduces to
# +/- V_interior / h depending on which face is being evaluated.
# sigma_ind = -eps_0 * eps_r * dV/dn_outward.
sigma_top    = -eps_0 * eps_r * (V_total_final[:, :,  0] - 0.0) / h  # n = -z
sigma_bottom = -eps_0 * eps_r * (V_total_final[:, :, -1] - 0.0) / h  # n = +z
sigma_xneg   = -eps_0 * eps_r * (V_total_final[ 0, :, :] - 0.0) / h  # n = -x
sigma_xpos   = -eps_0 * eps_r * (V_total_final[-1, :, :] - 0.0) / h  # n = +x
sigma_yneg   = -eps_0 * eps_r * (V_total_final[:,  0, :] - 0.0) / h  # n = -y
sigma_ypos   = -eps_0 * eps_r * (V_total_final[:, -1, :] - 0.0) / h  # n = +y

peak = max(
    np.abs(sigma_top).max(),
    np.abs(sigma_bottom).max(),
    np.abs(sigma_xneg).max(),
    np.abs(sigma_xpos).max(),
    np.abs(sigma_yneg).max(),
    np.abs(sigma_ypos).max(),
)
print(
    f"Peak |sigma_ind| across all six walls:"
    f" {peak:.3e} C/m^2  ({peak * 1e6:.3f} uC/m^2)"
)

### Step 8.2 — Per-Face Charge and the Gauss-Law Check

Integrating each $\sigma_\text{ind}$ array over its face gives the total induced charge sitting on that wall. The grand total across all six walls must, by Gauss's law applied to the closed box, cancel whatever interior free charge remains:

$$\sum_\text{faces} Q_\text{ind} \;+\; Q_\text{interior,\ remaining} \;=\; 0.$$

This is a strong numerical check on the whole pipeline — if it holds to a few parts in $10^4$, the Poisson solve, the tree-BC relaxation, and the charge-absorption bookkeeping are all internally consistent. The test fails visibly if the final Poisson re-solve was skipped (so $V_\text{free}$ still corresponds to the original $\rho$ rather than the depleted one) or if $V_\text{induced}$ isn't fully converged on the tree.

The per-face breakdown is informative too: it shows *how* the induced charge redistributes across the enclosure, which depends on where the charge cloud sits and how much of it the tree has swept up.

In [ ]:
# Step 8.2 — Integrate each face and verify Gauss's law.

dA = (resolution * cm_to_m) ** 2

Q_top    = sigma_top.sum()    * dA
Q_bottom = sigma_bottom.sum() * dA
Q_xneg   = sigma_xneg.sum()   * dA
Q_xpos   = sigma_xpos.sum()   * dA
Q_yneg   = sigma_yneg.sum()   * dA
Q_ypos   = sigma_ypos.sum()   * dA

Q_ind_total = Q_top + Q_bottom + Q_xneg + Q_xpos + Q_yneg + Q_ypos
Q_interior_remaining = rho_remaining.sum() * dV_cell_m3

print("Induced charge per face"
      " (negative = electrons piled on that wall):")
print(
    f"  top    (z=0)           :"
    f" {Q_top * 1e6:+7.3f} uC"
    f"  ({Q_top / Q_ind_total * 100:5.1f}%)"
)
print(
    f"  bottom (z={z_size})           :"
    f" {Q_bottom * 1e6:+7.3f} uC"
    f"  ({Q_bottom / Q_ind_total * 100:5.1f}%)"
)
print(
    f"  x = 0                  :"
    f" {Q_xneg * 1e6:+7.3f} uC"
    f"  ({Q_xneg / Q_ind_total * 100:5.1f}%)"
)
print(
    f"  x = {x_size}                 :"
    f" {Q_xpos * 1e6:+7.3f} uC"
    f"  ({Q_xpos / Q_ind_total * 100:5.1f}%)"
)
print(
    f"  y = 0                  :"
    f" {Q_yneg * 1e6:+7.3f} uC"
    f"  ({Q_yneg / Q_ind_total * 100:5.1f}%)"
)
print(
    f"  y = {y_size}                 :"
    f" {Q_ypos * 1e6:+7.3f} uC"
    f"  ({Q_ypos / Q_ind_total * 100:5.1f}%)"
)
print(f"  total induced          : {Q_ind_total * 1e6:+7.3f} uC")
print(f"Remaining interior charge: {Q_interior_remaining * 1e6:+7.3f} uC")
print(
    f"Sum (~0 by Gauss)        :"
    f" {(Q_ind_total + Q_interior_remaining) * 1e6:+.3e} uC"
)

## 9. Aftermath (b) — The Tree in 3D (with Growth Slider)

Interactive figure, rendered in Plotly with a slider and Play/Pause buttons so the discharge can be replayed step by step.

- **Drag the slider** to scrub through growth history — every position shows the tree as it existed at that growth step, with older branches and newer tips correctly separated.
- **Play** animates the full history at ~12 frames per second; **Pause** stops on the current frame.
- **Branches** are drawn parent-to-child, using the 26-connectivity edge that actually won the probability draw in the DBM loop (recorded in `parent_map` at growth time).
- **Tree cells** are coloured by growth step on the Inferno scale; the colour range is fixed over the whole animation so a dark cell at step 50 looks the same in frame 200 as it does in frame 3000.
- **Space charge** $\rho$ is drawn as a translucent blue volume and does *not* change across frames — we only animate the tree.
- **The nail** is marked with a cyan diamond.

Rotate the view while playing to watch branching structures fan out from the side face and propagate laterally through the charge disk.

**The three steps below:**

1. Build a per-frame trace generator — a single function that, given a growth-step cutoff, returns the edges and cells that should be drawn at that cutoff.
2. Assemble the frame list plus the static scene elements (charge volume, nail).
3. Wire up the slider, Play/Pause buttons, and assemble the final figure.

### Step 9.1 — Per-Frame Trace Builder

The animation is implemented by assembling ~40 Plotly frames, each showing the tree as it existed at a particular growth-step cutoff. A single helper function, `build_frame_traces(s_max)`, produces the two *animated* traces (branches + tree-cell markers) for any cutoff; we reuse it both to populate every frame and to construct the figure's initial data.

Design choices inside the function:

- **Cells** are selected with `(tree_step >= 0) & (tree_step <= s_max)`. This picks up the nail (step 0) and every cell added by step `s_max`. Cells with `tree_step = -1` (never added) are excluded.
- **Edges** are filtered by the *child* cell's growth step — if the child was added by `s_max`, the edge to its parent is valid (the parent was necessarily added earlier).
- **Colour range is pinned** globally via `cmin = 0, cmax = max_step_val`. Without this, the Inferno colourbar would rescale in every frame, so a step-42 cell would look different early in the animation than at the end.

We also cache the `parent_map` as a flat list of `(child, parent)` tuples once, so the per-frame loop can iterate without dict overhead.

In [ ]:
# Step 9.1 — Per-frame trace builder.

# Largest growth step actually reached in this run.
max_step_val = int(tree_step[tree].max())

# Flatten parent_map for fast per-frame iteration.
parent_pairs = list(parent_map.items())


def build_frame_traces(s_max):
    """Return (edge_trace, cell_trace) showing the tree up to step s_max."""
    vis_mask = (tree_step >= 0) & (tree_step <= s_max)
    cells = np.argwhere(vis_mask)
    cx = cells[:, 0] * resolution
    cy = cells[:, 1] * resolution
    cz = cells[:, 2] * resolution
    csteps_vis = tree_step[cells[:, 0], cells[:, 1], cells[:, 2]]

    ex, ey, ez = [], [], []
    for child, parent in parent_pairs:
        if tree_step[child] <= s_max:
            ex += [child[0] * resolution, parent[0] * resolution, None]
            ey += [child[1] * resolution, parent[1] * resolution, None]
            ez += [child[2] * resolution, parent[2] * resolution, None]

    edge_trace = go.Scatter3d(
        x=ex, y=ey, z=ez,
        mode="lines",
        line=dict(color="orange", width=2),
        hoverinfo="skip",
        name="branches",
    )
    cell_trace = go.Scatter3d(
        x=cx, y=cy, z=cz,
        mode="markers",
        marker=dict(
            size=2,
            color=csteps_vis,
            colorscale="Inferno",
            cmin=0,
            cmax=max_step_val,
            colorbar=dict(title="growth step"),
        ),
        name="tree cells",
        hovertemplate="step %{marker.color}<extra></extra>",
    )
    return edge_trace, cell_trace


print(f"Final growth step reached: {max_step_val}")
print(f"Parent-child edges to animate: {len(parent_pairs)}")

### Step 9.2 — Frame Schedule and Static Scene Elements

Two more pieces before we can assemble the figure.

**Frame schedule.** We pick 40 evenly-spaced growth-step cutoffs from 0 to `max_step_val`. `np.linspace(...).astype(int)` followed by `set()` deduplicates any repeats that show up if the run was short and several cutoffs round to the same integer. Each frame's `data` argument provides the two animated traces from `build_frame_traces`; `traces=[1, 2]` tells Plotly to only update the branches and cells when the frame is shown, leaving the charge volume and the nail untouched.

**Static traces.** The charge-density volume is built from `rho_remaining` (the *final* depleted state), so even when you scrub the slider back to step 0 the blue cloud already has gaps where the tree will eventually sweep. That's a deliberate choice — it's clearer to see "the tree is growing into charge that currently exists but will be cleared" than to animate the cloud too. The nail is a single-point cyan diamond.

At the end of this block we also pre-build the *final-state* edge and cell traces (the slider defaults to the last frame, so the notebook opens on the fully-grown tree).

In [ ]:
# Step 9.2 — Frame schedule and static scene elements.

# ~40 evenly-spaced cutoffs; set() deduplicates when the run was short.
n_frames = 40
frame_steps = sorted(set(
    np.linspace(0, max_step_val, n_frames, dtype=int).tolist()
))

# Build one frame per cutoff.  traces=[1, 2] says "update the branch
# trace (index 1) and the cell trace (index 2) only" -- the volume
# (index 0) and the nail (index 3) persist unchanged across frames.
frames = []
for s in frame_steps:
    edge_tr, cell_tr = build_frame_traces(int(s))
    frames.append(go.Frame(
        name=str(s),
        data=[edge_tr, cell_tr],
        traces=[1, 2],
    ))

# Static trace 0 -- space-charge volume (final rho_remaining).
rho_mag = np.abs(rho_remaining)
rho_max = float(rho_mag.max()) if rho_mag.max() > 0 else 1.0

volume_trace = go.Volume(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=(rho_mag / rho_max).flatten(),
    isomin=0.05, isomax=1.0,
    opacity=0.08, surface_count=12,
    colorscale="Blues",
    showscale=False,
    name="space charge |rho|",
    hoverinfo="skip",
)

# Static trace 3 -- the nail.
nail_xyz = (
    nail_idx[0] * resolution,
    nail_idx[1] * resolution,
    nail_idx[2] * resolution,
)
nail_trace = go.Scatter3d(
    x=[nail_xyz[0]], y=[nail_xyz[1]], z=[nail_xyz[2]],
    mode="markers",
    marker=dict(size=7, color="cyan", symbol="diamond"),
    name="nail",
)

# Final-state traces (used as the figure's initial data).
final_edge_tr, final_cell_tr = build_frame_traces(max_step_val)

print(
    f"Built {len(frames)} frames"
    f" covering steps {frame_steps[0]} .. {frame_steps[-1]}"
)

### Step 9.3 — Slider, Play/Pause, and Figure Assembly

Last block. Three Plotly widgets and the figure itself.

**Slider.** One entry per frame. Dragging to a given entry fires Plotly's `animate` command targeting that frame's name (`str(s)`). `duration: 0, redraw: True, mode: "immediate"` means the change is instantaneous — no interpolation between positions, which keeps scrubbing responsive. `currentvalue.prefix` makes the slider label show "growth step = N" above the track.

**Play / Pause buttons.** Implemented as Plotly `updatemenus` above the figure.

- *Play* calls `animate` with `fromcurrent: True` (resume from wherever the slider sits rather than restarting) and a frame duration of 80 ms — about 12 fps, slow enough to see the tree grow but fast enough to not be tedious.
- *Pause* calls `animate` with a null target list and `duration: 0`, which halts any in-progress animation at the current frame.

**Initial figure.** The figure's `data` argument gets four traces in exactly this order: volume (index 0), final edge trace (1), final cell trace (2), nail (3). The slider's `active` index is set to the last frame, so on load the user sees the completed tree and can scrub backward to replay. The 3D scene uses `aspectmode = "data"` so the axis scales reflect the physical 10 × 10 × 4 cm ratio rather than being stretched to a cube.

In [ ]:
# Step 9.3 — Slider, Play/Pause, and figure assembly.

# Slider: one entry per frame, immediate redraw on scrub.
slider_steps = [
    dict(
        method="animate",
        label=str(s),
        args=[
            [str(s)],
            dict(
                frame=dict(duration=0, redraw=True),
                mode="immediate",
                transition=dict(duration=0),
            ),
        ],
    )
    for s in frame_steps
]

sliders = [dict(
    active=len(frame_steps) - 1,
    currentvalue=dict(prefix="growth step = "),
    pad=dict(b=10, t=30),
    len=0.9,
    x=0.05,
    y=0,
    steps=slider_steps,
)]

# Play / Pause buttons.
play_args = [
    None,
    dict(
        frame=dict(duration=80, redraw=True),
        fromcurrent=True,
        transition=dict(duration=0),
    ),
]
pause_args = [
    [None],
    dict(
        frame=dict(duration=0, redraw=False),
        mode="immediate",
        transition=dict(duration=0),
    ),
]

updatemenus = [dict(
    type="buttons",
    direction="left",
    showactive=False,
    x=0.05,
    y=1.05,
    xanchor="left",
    yanchor="top",
    pad=dict(t=0, r=10),
    buttons=[
        dict(label="Play", method="animate", args=play_args),
        dict(label="Pause", method="animate", args=pause_args),
    ],
)]

# Assemble.  Trace order: [volume, edges, cells, nail]; frames only
# replace traces 1 and 2 (edges, cells).
fig3d = go.Figure(
    data=[volume_trace, final_edge_tr, final_cell_tr, nail_trace],
    frames=frames,
)
fig3d.update_layout(
    scene=dict(
        xaxis=dict(title="x (cm)", range=[0, x_size]),
        yaxis=dict(title="y (cm)", range=[0, y_size]),
        zaxis=dict(title="z (cm)", range=[0, z_size]),
        aspectmode="data",
    ),
    title=(
        f"Lichtenberg discharge (defined rho) --"
        f" eta={eta}, 26-conn,"
        f" {int(tree.sum())} tree cells,"
        f" {Q_absorbed_running * 1e6:+.2f} uC absorbed"
    ),
    margin=dict(l=0, r=0, b=90, t=60),
    width=950,
    height=820,
    updatemenus=updatemenus,
    sliders=sliders,
)
fig3d.show()

### Step 9.4 — Export the Animation as a GIF

The Plotly slider figure above is interactive but only lives inside an active kernel. To share the animation as a standalone artefact we re-render each frame with matplotlib's 3D projection, save it as an in-memory PNG, then stitch the sequence together with Pillow into an animated GIF.

Why not use Plotly's own image export? It requires the `kaleido` backend, which isn't always installed. Matplotlib is already available, and its `Line3DCollection` batches all parent→child edges of a frame into a single artist — much faster than calling `ax.plot` thousands of times per frame.

The output is written to `lichtenberg_discharge.gif` in the notebook directory at ~12 frames per second (80 ms per frame, matching the Play button cadence). Typical file size is 2–5 MB for a 40-frame run of a few thousand cells. Generation takes roughly 20–30 seconds on the default grid.

One difference from the Plotly figure: we skip the translucent charge cloud in the GIF (volumetric rendering per frame in matplotlib is too slow to be worthwhile) and rely on the tree + nail alone. The `view_init` angle is fixed so the camera doesn't drift between frames.

In [ ]:
# Step 9.4 — Export the growth animation as a GIF.

gif_path = "lichtenberg_discharge.gif"
frame_duration_ms = 80
gif_dpi = 90

png_frames = []
step_norm = Normalize(vmin=0, vmax=max_step_val)

for s in frame_steps:
    fig_gif = plt.figure(figsize=(7, 5))
    ax = fig_gif.add_subplot(111, projection="3d")

    # Tree cells added by step s.
    mask = (tree_step >= 0) & (tree_step <= s)
    cells = np.argwhere(mask)
    if len(cells):
        cx = cells[:, 0] * resolution
        cy = cells[:, 1] * resolution
        cz = cells[:, 2] * resolution
        ctimes = tree_step[cells[:, 0], cells[:, 1], cells[:, 2]]
        ax.scatter(
            cx, cy, cz,
            c=ctimes,
            cmap="inferno",
            vmin=0, vmax=max_step_val,
            s=2, depthshade=False,
        )

    # Parent->child edges up to step s, batched into one Line3DCollection.
    segments = [
        [
            (child[0] * resolution,
             child[1] * resolution,
             child[2] * resolution),
            (parent[0] * resolution,
             parent[1] * resolution,
             parent[2] * resolution),
        ]
        for child, parent in parent_pairs
        if tree_step[child] <= s
    ]
    if segments:
        lc = Line3DCollection(
            segments, colors="orange", linewidths=0.25, alpha=0.9,
        )
        ax.add_collection3d(lc)

    # Nail marker.
    ax.scatter(
        [nail_xyz[0]], [nail_xyz[1]], [nail_xyz[2]],
        color="cyan", s=60, marker="D",
        edgecolors="black", linewidths=0.8, depthshade=False,
    )

    ax.set_xlim(0, x_size)
    ax.set_ylim(0, y_size)
    ax.set_zlim(0, z_size)
    ax.set_xlabel("x (cm)")
    ax.set_ylabel("y (cm)")
    ax.set_zlabel("z (cm)")
    ax.set_box_aspect((x_size, y_size, z_size))
    ax.view_init(elev=22, azim=45)
    ax.set_title(f"growth step = {s} / {max_step_val}")

    buf = io.BytesIO()
    fig_gif.savefig(buf, format="png", dpi=gif_dpi)
    plt.close(fig_gif)
    buf.seek(0)
    png_frames.append(Image.open(buf).convert("RGB"))

# Enforce a common size (matplotlib should already do this, but guard).
target_size = png_frames[0].size
png_frames = [
    f if f.size == target_size else f.resize(target_size)
    for f in png_frames
]

png_frames[0].save(
    gif_path,
    save_all=True,
    append_images=png_frames[1:],
    duration=frame_duration_ms,
    loop=0,
    optimize=True,
)
print(f"Saved {len(png_frames)}-frame GIF to: {gif_path}")
print(f"Frame size: {target_size[0]} x {target_size[1]} px"
      f"  |  duration: {frame_duration_ms} ms/frame"
      f"  |  total: {len(png_frames) * frame_duration_ms / 1000:.1f} s")

### Step 9.5 — Top-Down GIF (xy-Plane View)

The 3D GIF above makes the depth of the tree clear but obscures the lateral branching pattern. A second GIF rendered from directly above — camera looking straight down the $z$-axis onto the $xy$-plane — shows the disk-plane spread much more clearly.

This view uses a plain 2D matplotlib axis instead of the 3D projection (faster to render and sharper at equal pixel budget). Depth is encoded only in the step-time colour; every branch is projected onto the $xy$-plane, so crossings that aren't physically touching in 3D may appear to overlap in 2D. That's expected.

The output is written to `lichtenberg_discharge_topdown.gif` alongside the oblique-view GIF.

In [ ]:
# Step 9.5 — Top-down (xy-plane) GIF.

gif_path_top = "lichtenberg_discharge_topdown.gif"
png_frames_top = []

for s in frame_steps:
    fig_top, ax = plt.subplots(figsize=(6, 6))

    # Tree cells added by step s, coloured by addition time.
    mask = (tree_step >= 0) & (tree_step <= s)
    cells = np.argwhere(mask)
    if len(cells):
        cx = cells[:, 0] * resolution
        cy = cells[:, 1] * resolution
        ctimes = tree_step[cells[:, 0], cells[:, 1], cells[:, 2]]
        ax.scatter(
            cx, cy,
            c=ctimes, cmap="inferno",
            vmin=0, vmax=max_step_val,
            s=2,
        )

    # Parent->child edges up to step s, projected to xy.
    segments_2d = [
        [
            (child[0] * resolution, child[1] * resolution),
            (parent[0] * resolution, parent[1] * resolution),
        ]
        for child, parent in parent_pairs
        if tree_step[child] <= s
    ]
    if segments_2d:
        lc2d = LineCollection(
            segments_2d, colors="orange", linewidths=0.25, alpha=0.9,
        )
        ax.add_collection(lc2d)

    # Nail projected to xy.
    ax.scatter(
        [nail_xyz[0]], [nail_xyz[1]],
        color="cyan", s=60, marker="D",
        edgecolors="black", linewidths=0.8,
    )

    ax.set_xlim(0, x_size)
    ax.set_ylim(0, y_size)
    ax.set_aspect("equal")
    ax.set_xlabel("x (cm)")
    ax.set_ylabel("y (cm)")
    ax.set_title(f"top-down view  |  growth step = {s} / {max_step_val}")

    buf = io.BytesIO()
    fig_top.savefig(buf, format="png", dpi=gif_dpi, bbox_inches="tight")
    plt.close(fig_top)
    buf.seek(0)
    png_frames_top.append(Image.open(buf).convert("RGB"))

target_size_top = png_frames_top[0].size
png_frames_top = [
    f if f.size == target_size_top else f.resize(target_size_top)
    for f in png_frames_top
]

png_frames_top[0].save(
    gif_path_top,
    save_all=True,
    append_images=png_frames_top[1:],
    duration=frame_duration_ms,
    loop=0,
    optimize=True,
)
print(f"Saved {len(png_frames_top)}-frame top-down GIF to: {gif_path_top}")
print(
    f"Frame size: {target_size_top[0]} x {target_size_top[1]} px"
    f"  |  duration: {frame_duration_ms} ms/frame"
    f"  |  total: {len(png_frames_top) * frame_duration_ms / 1000:.1f} s"
)

## 10. Aftermath (c) — Fractal Morphology

Final question: does the tree we grew actually look like a real Lichtenberg figure? We project the 3D tree top-down (collapse the $z$ axis) to get the 2D "photograph" a viewer would see looking into the slab, then extract two scale-invariant diagnostics.

- **Box-counting dimension $D_f$** (Theory §1.1) — count occupied boxes of side $\epsilon$ across a range of scales, fit the slope of $\ln N(\epsilon)$ vs. $\ln(1/\epsilon)$. Reference range for real charge-implanted PMMA figures (Theory §4.5): $D_f \approx 1.65$ – $1.75$.
- **Lacunarity $\Lambda(\epsilon)$** (Theory §1.3) — gliding-box variance of occupation mass at scale $\epsilon$. Quantifies *textural* heterogeneity: two trees with identical $D_f$ can have very different $\Lambda$, distinguishing "evenly sparse" from "bunchy and gappy" at fixed mean density.

If $D_f$ lands outside the expected band, the usual suspects trace back to the discharge parameters:

- $\eta$ too low → tree fills like a blob instead of branching like a dendrite.
- $\beta_\text{tip}$ mis-set → the effective breakdown threshold is wrong, so the tree grows too eagerly or stalls too early.
- `sigma_fraction` too small → insufficient field energy to drive a big tree in the first place, so the stopping condition hits before a good fractal regime has developed.

That's the end of the story — from material constants at the top to a quantitative comparison with real Lichtenberg figures at the bottom.

In [ ]:
# Top-down projection of the 3D tree.
tree_2d = tree.any(axis=2)


def box_count(binary, box_sizes):
    """Count boxes of side s that contain at least one occupied cell."""
    counts = []
    H, W = binary.shape
    for s in box_sizes:
        Hs, Ws = (H // s) * s, (W // s) * s
        block = binary[:Hs, :Ws].reshape(Hs // s, s, Ws // s, s)
        counts.append(int(block.any(axis=(1, 3)).sum()))
    return np.array(counts)


def lacunarity(binary, s):
    """Gliding-box lacunarity at scale s, computed via an integral image."""
    H, W = binary.shape
    if s >= min(H, W):
        return np.nan
    ii = np.pad(
        np.cumsum(np.cumsum(binary.astype(np.int64), 0), 1),
        ((1, 0), (1, 0)),
    )
    mass = (
        ii[s:, s:] - ii[:-s, s:] - ii[s:, :-s] + ii[:-s, :-s]
    ).astype(float)
    m1 = mass.mean()
    if m1 <= 0:
        return np.nan
    return (mass ** 2).mean() / (m1 ** 2)


max_box = max(1, min(tree_2d.shape) // 3)
box_sizes = np.unique(
    np.round(np.logspace(0, np.log10(max_box), 10)).astype(int)
)
counts = box_count(tree_2d, box_sizes)

mask = counts > 0
log_s = np.log(box_sizes[mask])
log_N = np.log(counts[mask])
slope, intercept = np.polyfit(log_s, log_N, 1)
D_f = -slope

lac = np.array([lacunarity(tree_2d, s) for s in box_sizes])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(
    tree_2d.T, origin="lower", extent=[0, x_size, 0, y_size],
    cmap="inferno", interpolation="nearest",
)
axes[0].plot(
    nail_idx[0] * resolution, nail_idx[1] * resolution,
    "co", markersize=6, label="nail",
)
axes[0].set(
    xlabel="x (cm)", ylabel="y (cm)", title="Top-down projection",
)
axes[0].legend(loc="upper right")

axes[1].loglog(
    box_sizes[mask], counts[mask],
    "o", color="steelblue", label="N(epsilon)",
)
fit_N = np.exp(intercept) * box_sizes[mask] ** slope
axes[1].loglog(
    box_sizes[mask], fit_N, "--", color="black",
    label=f"$D_f$ = {D_f:.3f}",
)
axes[1].set(
    xlabel="box size epsilon (px)", ylabel="N(epsilon)",
    title="Box-counting dimension",
)
axes[1].legend()

lac_mask = ~np.isnan(lac)
axes[2].loglog(
    box_sizes[lac_mask], lac[lac_mask], "s-", color="darkred",
)
axes[2].set(
    xlabel="box size epsilon (px)", ylabel="Lambda(epsilon)",
    title="Lacunarity",
)

plt.tight_layout()
plt.show()

print(f"Box-counting fractal dimension (2D projection):  D_f = {D_f:.3f}")
print("Expected range for acrylic Lichtenberg figures:  1.65 - 1.75")